# **final**

In [ ]:
%pip uninstall transformers -y
%pip install -r requirements.txt

In [ ]:
%pip install google-auth-oauthlib google-api-python-client

In [ ]:
import os
BASE_DIR = './Images' # Directory containing Images
UPLOAD_DIR = f'{BASE_DIR}/uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)

In [ ]:
import pandas as pd

ITEMS_FILE = f'{BASE_DIR}/items.json'

COLUMNS = ['id', 'image_path', 'category', 'subcategory',
           'color_primary', 'color_secondary', 'warmth',
           'formality', 'waterproof', 'last_worn_at']

if os.path.exists(ITEMS_FILE):
    items_df = pd.read_json(ITEMS_FILE)
else:
    items_df = pd.DataFrame(columns=COLUMNS)


In [ ]:
items_df

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

In [ ]:
import torch, json, re
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

florence_model = AutoModelForCausalLM.from_pretrained(
    "anushreeberlia/fashion-florence",
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device)

florence_processor = AutoProcessor.from_pretrained(
    "anushreeberlia/fashion-florence", trust_remote_code=True,
)

In [ ]:
def run_florence_tagging(image_path):
    image = Image.open(image_path).convert("RGB")
    image.thumbnail((768, 768), Image.Resampling.BILINEAR)
    prompt = (
        "Analyze this clothing image and return ONLY valid JSON in this exact shape: "
        "{\"category\": \"top|bottom|outerwear|shoes|accessory\", "
        "\"primary_color\": \"color\", "
        "\"material\": \"material\", "
        "\"style_tags\": [\"tag1\", \"tag2\"]}"
    )

    inputs = florence_processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    print("Using max_new_tokens=", 200, "num_beams=", 1)  # match whatever you actually pass
    with torch.no_grad():
        generated_ids = florence_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=200,
            num_beams=1,
        )
    result = florence_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return result

In [ ]:
import re
import json

def sanitize_model_json(raw_output):
    text = str(raw_output).strip()

    text = re.sub(r"```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found in output: {text[:200]}")

    text = text[start:end + 1]

    fixes = [
        # Remove trailing commas before } or ]
        (r",\s*([}\]])", r"\1"),  
        
        # Quote unquoted keys: {category: -> {"category":
        (r"([\{,]\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1"\2":'),  
        
        # Quote single-quoted string values: : 'top' -> : "top"
        (r":\s*'([^']*)'(?=\s*[,}\]])", r': "\1"'),  
        
        # Replace Python types with JSON types
        (r"\bNone\b", "null"),
        (r"\bTrue\b", "true"),
        (r"\bFalse\b", "false"),
        
        # Quote unquoted string values: : top, -> : "top",
        (r":\s*([A-Za-z_][A-Za-z0-9_]*)\s*(?=[,}\]])", r': "\1"'),
    ]

    for pattern, repl in fixes:
        text = re.sub(pattern, repl, text)

    # This fixes the exact typo the model keeps making
    text = re.sub(r'("material"\s*:\s*"[^"]*)\{("style_tags")', r'\1",\2', text)

    return text

In [ ]:
def parse_florence_json(raw_output):
    match = re.search(r"\{.*\}", raw_output, re.DOTALL)
    if not match:
        print(f"⚠️ No JSON structure found: {raw_output}")
        return {"category": "top", "primary_color": "unknown", "material": "unknown", "style_tags": []}

    candidate = match.group()
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        print(f"⚠️ Malformed JSON, using fallback: {raw_output}")
        return {"category": "top", "primary_color": "unknown", "material": "unknown", "style_tags": []}

In [ ]:
# ============================================================
# CELL — Color detection (Lab-space + neutral-aware)
# ============================================================
import numpy as np
import colorsys
from PIL import Image
from rembg import remove
from sklearn.cluster import KMeans
from skimage.color import rgb2lab

NAMED_COLORS_RGB = {
    "black": (0, 0, 0),
    "white": (255, 255, 255),
    "grey": (128, 128, 128),
    "charcoal": (54, 54, 54),
    "light-grey": (200, 200, 200),
    "beige": (222, 202, 173),
    "cream": (240, 234, 214),
    "tan": (210, 180, 140),
    "khaki": (195, 176, 145),
    "navy": (0, 0, 80),
    "blue": (30, 90, 200),
    "light-blue": (140, 190, 230),
    "denim": (70, 100, 140),
    "teal": (0, 128, 128),
    "green": (40, 130, 60),
    "olive": (100, 110, 40),
    "sage": (150, 165, 130),
    "mint": (170, 220, 190),
    "red": (200, 30, 30),
    "maroon": (110, 20, 30),
    "burgundy": (90, 20, 35),
    "pink": (230, 150, 180),
    "coral": (240, 120, 100),
    "yellow": (220, 200, 40),
    "mustard": (200, 160, 40),
    "orange": (230, 120, 30),
    "rust": (160, 80, 40),
    "purple": (110, 60, 140),
    "lavender": (190, 170, 220),
    "brown": (110, 70, 40),
    "chocolate": (70, 40, 25),
    "camel": (170, 120, 70),
    "dusty-rose": (196, 111, 117),
    "mauve": (150, 100, 110),
    "terracotta": (200, 100, 70),
    "salmon": (230, 140, 130),
    "wine": (114, 47, 55),
}

NEUTRAL_NAMES = {"black", "white", "grey", "charcoal", "light-grey"}

def rgb_to_lab(rgb):
    arr = np.array([[rgb]], dtype=np.uint8) / 255.0
    return rgb2lab(arr)[0][0]

NAMED_COLORS_LAB = {name: rgb_to_lab(rgb) for name, rgb in NAMED_COLORS_RGB.items()}
CHROMATIC_COLORS_LAB = {k: v for k, v in NAMED_COLORS_LAB.items() if k not in NEUTRAL_NAMES}

def rgb_to_hsv(rgb):
    r, g, b = [c / 255.0 for c in rgb]
    return colorsys.rgb_to_hsv(r, g, b)

def classify_neutral(rgb):
    _, s, v = rgb_to_hsv(rgb)
    if s > 0.15:
        return None
    if v > 0.85:
        return "white"
    elif v > 0.65:
        return "light-grey"
    elif v > 0.45:
        return "grey"
    elif v > 0.25:
        return "charcoal"
    else:
        return "black"

def nearest_named_color(rgb):
    neutral = classify_neutral(rgb)
    if neutral:
        return neutral
    target_lab = rgb_to_lab(rgb)
    def distance(lab1, lab2):
        return sum((a - b) ** 2 for a, b in zip(lab1, lab2))
    return min(CHROMATIC_COLORS_LAB, key=lambda name: distance(target_lab, CHROMATIC_COLORS_LAB[name]))

def remove_background(image_path):
    img = Image.open(image_path).convert("RGBA")
    return remove(img)

def get_dominant_color(rgba_image, k=3):
    arr = np.array(rgba_image)
    mask = arr[:, :, 3] > 128
    pixels = arr[mask][:, :3]
    if len(pixels) == 0:
        return (0, 0, 0)
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(pixels)
    counts = np.bincount(kmeans.labels_)
    dominant = kmeans.cluster_centers_[np.argmax(counts)]
    return tuple(int(c) for c in dominant)

def extract_color(image_path, k=3):
    bg_removed = remove_background(image_path)
    dominant_rgb = get_dominant_color(bg_removed, k=k)
    return nearest_named_color(dominant_rgb)

In [ ]:
ALLOWED_CATEGORIES = {"top", "bottom","dress", "outerwear", "shoes", "accessory"}

WARM_MATERIALS = {"wool", "fleece", "down", "leather", "denim"}
LIGHT_MATERIALS = {"cotton", "linen", "silk", "polyester-mesh"}

FORMAL_STYLE_KEYWORDS = {"blazer", "suit", "dress-shirt", "trouser", "formal"}
ATHLETIC_STYLE_KEYWORDS = {"sport", "athletic", "gym", "activewear", "track"}

def infer_warmth(material, category):
    material = (material or "").lower()
    if material in WARM_MATERIALS or category == "outerwear":
        return "heavy"
    if material in LIGHT_MATERIALS:
        return "light"
    return "medium"  # covers "unknown" material too

def infer_formality(style_tags):
    text = " ".join(style_tags or []).lower()
    if any(k in text for k in FORMAL_STYLE_KEYWORDS):
        return "formal"
    if any(k in text for k in ATHLETIC_STYLE_KEYWORDS):
        return "athletic"
    return "casual"

def map_to_schema(florence_output):
    category = florence_output.get("category", "top").lower()
    if category not in ALLOWED_CATEGORIES:
        category = "top"

    material = florence_output.get("material")
    style_tags = florence_output.get("style_tags", [])

    return {
        "category": category,
        "subcategory": style_tags[0] if style_tags else "",
        "color_primary": None,  # placeholder — always overwritten by extract_color()
        "warmth": infer_warmth(material, category),
        "formality": infer_formality(style_tags),
        "waterproof": ("waterproof" in style_tags) or ("rain" in style_tags),
    }

In [ ]:
def auto_tag_item(image_path):
    raw_output = run_florence_tagging(image_path)
    print(raw_output)
    parsed = parse_florence_json(raw_output)
    tags = map_to_schema(parsed)

    tags["color_primary"] = extract_color(image_path)  # k-means + Lab + neutral detection
    return tags

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import shutil, uuid

In [ ]:
# ============================================================
# CELL 5e-alt — List images already in Drive
# ============================================================
import os

def list_drive_images(folder=UPLOAD_DIR):
    valid_ext = (".jpg", ".jpeg", ".png")
    files = [f for f in os.listdir(folder) if f.lower().endswith(valid_ext)]
    return sorted(files)

available_images = list_drive_images()
print(f"Found {len(available_images)} images")

In [ ]:
def list_untagged_images(folder=UPLOAD_DIR):
    all_images = list_drive_images(folder)
    already_tagged = {os.path.basename(p) for p in items_df["image_path"]}
    return [f for f in all_images if f not in already_tagged]

In [ ]:
# ============================================================
# CELL 5f — Upload widget with auto-tag pre-fill (Drive picker + preview)
# ============================================================
import uuid
from PIL import Image
import io

image_picker = widgets.Dropdown(
    options=list_untagged_images(),
    description='Image:'
)
refresh_btn = widgets.Button(description='Refresh list')
preview_output = widgets.Output()
run_autotag_btn = widgets.Button(description='Auto-tag')
status_output = widgets.Output()
category = widgets.Dropdown(options=['top','bottom','dress','outerwear','shoes','accessory'], description='Category:')
subcategory_input = widgets.Text(description='Subcategory:')
warmth = widgets.Dropdown(options=['light','medium','heavy'], description='Warmth:')
formality = widgets.Dropdown(options=['casual','smart-casual','formal','athletic'], description='Formality:')
color = widgets.Dropdown(options=list(NAMED_COLORS_RGB.keys()), description='Color:')
waterproof = widgets.Checkbox(description='Waterproof')
submit = widgets.Button(description='Confirm & add item')

display(image_picker, refresh_btn, preview_output, run_autotag_btn, status_output,
        category, subcategory_input, warmth, formality, color, waterproof, submit)

_current_image_path: dict[str, str | None] = {"path": None}

def show_preview(filename):
    preview_output.clear_output()
    if not filename:
        return
    path = os.path.join(UPLOAD_DIR, filename)
    with preview_output:
        img = Image.open(path)
        img.thumbnail((200, 200))
        display(img)

def on_image_change(change):
    if change["name"] == "value":
        show_preview(change["new"])

image_picker.observe(on_image_change, names="value")

def on_refresh(b):
    image_picker.options = list_untagged_images()
    with status_output:
        status_output.clear_output()
        print(f"Refreshed — {len(image_picker.options)} untagged images found")

def on_autotag(b):
    if not image_picker.value:
        with status_output:
            status_output.clear_output()
            print("No image selected")
        return

    run_autotag_btn.disabled = True
    with status_output:
        status_output.clear_output()
        print("⏳ Tagging in progress (usually ~60s)...")

    path = os.path.join(UPLOAD_DIR, image_picker.value)
    _current_image_path["path"] = path

    tags = auto_tag_item(path)
    category.value = tags["category"]
    subcategory_input.value = tags["subcategory"]
    warmth.value = tags["warmth"]
    formality.value = tags["formality"]
    color.value = tags["color_primary"] if tags["color_primary"] in color.options else "black"
    waterproof.value = tags["waterproof"]

    with status_output:
        status_output.clear_output()
        print("✅ Auto-tagged — review and correct below, then confirm.")
    run_autotag_btn.disabled = False

def on_submit(b):
    global items_df
    if not _current_image_path["path"]:
        with status_output:
            print("Run Auto-tag first")
        return

    new_row = {'id': str(uuid.uuid4())[:8], 'image_path': _current_image_path["path"],
               'category': category.value, 'subcategory': subcategory_input.value,
               'color_primary': color.value, 'color_secondary': None,
               'warmth': warmth.value, 'formality': formality.value,
               'waterproof': waterproof.value, 'last_worn_at': None}
    items_df = pd.concat([items_df, pd.DataFrame([new_row])], ignore_index=True)
    items_df.to_json(ITEMS_FILE, orient='records')

    with status_output:
        status_output.clear_output()
        print(f"Added item {new_row['id']}")

    image_picker.options = list_untagged_images()
    _current_image_path["path"] = None

refresh_btn.on_click(on_refresh)
run_autotag_btn.on_click(on_autotag)
submit.on_click(on_submit)

# Show preview for whatever's selected by default when the cell first runs
show_preview(image_picker.value)

In [ ]:


# Debugging / testing code

if image_picker.value:
    path = os.path.join(UPLOAD_DIR, image_picker.value)
    print("PATH:", path)
else:
    print("PATH: (dropdown empty — all images already tagged)")

TEST_IMG = "Images/uploads/Womens_casual_top2.jpg"

raw = run_florence_tagging(TEST_IMG)
print("RAW OUTPUT:", repr(raw))

result = auto_tag_item(TEST_IMG)
print("PARSED:", result)

raw2 = run_florence_tagging("Images/uploads/Womens_Formal_top6.jpg")
print("RAW OUTPUT 2:", repr(raw2))

In [ ]:
# Re-parse the most recent raw Florence output (set by cell 18 or cell 20)
try:
    parsed = parse_florence_json(raw)
    print(parsed)
except NameError:
    print("No raw output yet — run cell 18 or cell 20 first.")

In [ ]:
# ============================================================
# CELL 5g — Debug: see Florence's actual raw output
# ============================================================
test_path = "Images/uploads/women_jeans_2.jpg"  # or hardcode a path to any uploaded image
raw = run_florence_tagging(test_path)
print("RAW OUTPUT:")
print(raw)
print()

parsed = parse_florence_json(raw)
print("PARSED KEYS:")
print(parsed)

In [ ]:
import requests

def get_location_from_ip(test_ip=None):
    url = "http://ip-api.com/json/"
    if test_ip:
        url += test_ip
    response = requests.get(url)
    data = response.json()
    if data.get("status") == "fail":
        raise ValueError(f"Geolocation failed: {data.get('message')}")
    return {
        "lat": data["lat"],
        "lon": data["lon"],
        "city": data["city"],
        "country": data["country"]
    }

In [ ]:
from datetime import datetime

def fetch_hourly_weather(lat, lon, api_key):
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {"lat": lat, "lon": lon, "appid": api_key, "units": "metric"}
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

def get_weather_for_time(forecast_data, target_datetime):
    closest = min(
        forecast_data['list'],
        key=lambda entry: abs(datetime.fromtimestamp(entry['dt']) - target_datetime)
    )
    return {
        "temp": closest['main']['temp'],
        "feels_like": closest['main']['feels_like'],
        "rain_probability": closest.get('pop', 0) * 100,
        "condition": closest['weather'][0]['main'],
        "datetime": datetime.fromtimestamp(closest['dt'])
    }



In [ ]:
def build_occasion_time(hour, minute=0, base_date=None):
    base = base_date or datetime.now()
    return base.replace(hour=hour, minute=minute, second=0, microsecond=0)



In [ ]:
def map_temp_to_warmth(temp_celsius):
    if temp_celsius < 15:
        return "heavy"
    elif temp_celsius < 24:
        return "medium"
    else:
        return "light"

WARMTH_ORDER = ["light", "medium", "heavy"]

def warmth_is_acceptable(item_warmth, required_warmth, tolerance=1):
    item_idx = WARMTH_ORDER.index(item_warmth)
    required_idx = WARMTH_ORDER.index(required_warmth)
    return abs(item_idx - required_idx) <= tolerance


In [ ]:
OCCASION_FORMALITY = {
    "gym": "athletic",
    "office": "smart-casual",
    "dinner": "formal",
    "casual": "casual"
}

RAIN_THRESHOLD = 50

In [ ]:
def filter_items_for_occasion(items_df, occasion_type, weather):
    required_formality = OCCASION_FORMALITY[occasion_type]
    required_warmth = map_temp_to_warmth(weather["temp"])

    filtered = items_df[items_df["formality"] == required_formality].copy()

    if weather["rain_probability"] > RAIN_THRESHOLD:
        filtered = filtered[
            ~((filtered["category"] == "outerwear") & (~filtered["waterproof"]))
        ]

    warmth_relevant = filtered["category"].isin(["top", "bottom", "outerwear"])
    warmth_ok = filtered["warmth"].apply(lambda w: warmth_is_acceptable(w, required_warmth))
    filtered = filtered[~warmth_relevant | (warmth_relevant & warmth_ok)]

    return filtered


In [ ]:
from itertools import product

def split_by_category(filtered_df):
    return {
        "top": filtered_df[filtered_df["category"] == "top"].to_dict("records"),
        "bottom": filtered_df[filtered_df["category"] == "bottom"].to_dict("records"),
        "dress": filtered_df[filtered_df["category"] == "dress"].to_dict("records"),
        "outerwear": filtered_df[filtered_df["category"] == "outerwear"].to_dict("records"),
        "shoes": filtered_df[filtered_df["category"] == "shoes"].to_dict("records"),
    }

def generate_combos(categorized_items):
    tops = categorized_items["top"]
    bottoms = categorized_items["bottom"]
    dresses = categorized_items["dress"]
    outerwear_options = categorized_items["outerwear"] + [None]
    shoes = categorized_items["shoes"]

    missing = [cat for cat, items in [("top", tops), ("bottom", bottoms), ("dress", dresses), ("shoes", shoes)] if not items]
    if missing:
        print(f"Warning: no valid items for {missing} — cannot generate combos")
        return []

    combos = []
    for top, bottom, dress, outerwear, shoe in product(tops, bottoms, dresses, outerwear_options, shoes):
        combos.append({
            "top": top,
            "bottom": bottom,
            "dress": dress,
            "outerwear": outerwear,
            "shoes": shoe
        })
    return combos

In [ ]:
# ============================================================
# CELL 14 — Fetch today's occasions from Google Calendar
# ============================================================
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from datetime import datetime

SCOPES = ['https://www.googleapis.com/auth/calendar.readonly']

def get_calendar_service():
    flow = InstalledAppFlow.from_client_secrets_file('./credentials.json', SCOPES)
    creds = flow.run_local_server(port=0)
    return build('calendar', 'v3', credentials=creds)

def get_todays_events(service):
    now = datetime.utcnow()
    start_of_day = now.replace(hour=0, minute=0, second=0).isoformat() + 'Z'
    end_of_day = now.replace(hour=23, minute=59, second=59).isoformat() + 'Z'
    events_result = service.events().list(
        calendarId='primary', timeMin=start_of_day, timeMax=end_of_day,
        singleEvents=True, orderBy='startTime'
    ).execute()
    return events_result.get('items', [])

def guess_occasion_type(event_title):
    title = event_title.lower()
    if any(k in title for k in ["gym", "workout", "run"]):
        return "gym"
    elif any(k in title for k in ["dinner", "date"]):
        return "dinner"
    elif any(k in title for k in ["meeting", "standup", "office", "work"]):
        return "office"
    return "casual"

def parse_event_time(event):
    start_str = event['start'].get('dateTime')
    if not start_str:
        return None  # all-day event, has no specific time — skip for outfit purposes
    # strip timezone offset for compatibility with build_occasion_time-style datetimes
    return datetime.fromisoformat(start_str).replace(tzinfo=None)

def build_occasions_from_calendar():
    service = get_calendar_service()
    events = get_todays_events(service)

    occasions = {}
    for event in events:
        time = parse_event_time(event)
        if time is None:
            continue
        occasion_type = guess_occasion_type(event.get('summary', ''))
        occasions[occasion_type] = time  # last event of a given type wins, for now

    return occasions

occasions_dict = build_occasions_from_calendar()
print(occasions_dict)

In [ ]:
print(guess_occasion_type("Gym session"))       # expect "gym"
print(guess_occasion_type("Team standup"))      # expect "office"
print(guess_occasion_type("Dinner with Sarah")) # expect "dinner"
print(guess_occasion_type("Dentist appointment")) # expect "casual" — a real gap, worth knowing about

In [ ]:
location = get_location_from_ip(test_ip=None)
print("Location:", location)

forecast = fetch_hourly_weather(location["lat"], location["lon"], API_KEY)


In [ ]:
results = {}

for name, time in occasions_dict.items():
    weather = get_weather_for_time(forecast, time)
    filtered = filter_items_for_occasion(items_df, name, weather)
    categorized = split_by_category(filtered)
    combos = generate_combos(categorized)

    results[name] = {
       "weather": weather,
        "filtered_items": filtered,
        "combos": combos
    }

    print(f"\n{name}: {weather['temp']}°C, rain {weather['rain_probability']}% → {len(combos)} combos")

In [ ]:
NEUTRALS = {"black", "white", "grey", "beige", "navy"}

# Angular position on a simplified 12-color wheel
COLOR_WHEEL = {
    "red": 0, "orange": 1, "yellow": 2, "green": 3,
    "teal": 4, "blue": 5, "navy": 5, "purple": 6,
    "pink": 7, "brown": 8, "black": None, "white": None, "grey": None,
    "beige": None
}

WARM_COLORS = {"red", "orange", "yellow", "brown", "pink"}
COOL_COLORS = {"blue", "navy", "green", "teal", "purple"}

def color_compatibility_score(color_a, color_b):
    if color_a in NEUTRALS or color_b in NEUTRALS:
        return 1.0

    pos_a, pos_b = COLOR_WHEEL.get(color_a), COLOR_WHEEL.get(color_b)
    if pos_a is None or pos_b is None:
        return 0.6  # unknown color, default to a middling score

    distance = min(abs(pos_a - pos_b), 12 - abs(pos_a - pos_b))
    if distance <= 1:
        return 0.8   # analogous
    elif distance >= 5:
        return 0.9   # near-complementary
    elif (color_a in WARM_COLORS and color_b in COOL_COLORS) or \
         (color_a in COOL_COLORS and color_b in WARM_COLORS):
        return 0.4   # clashing warm/cool
    else:
        return 0.6   # same family, not neutral


In [ ]:
def combo_warmth_level(combo):
    levels = []
    for key in ["top", "bottom", "outerwear"]:
        item = combo[key]
        if item is not None:
            levels.append(WARMTH_ORDER.index(item["warmth"]))
    return sum(levels) / len(levels) if levels else 0

def temperature_fit_score(combo, weather):
    required_warmth = WARMTH_ORDER.index(map_temp_to_warmth(weather["temp"]))
    actual_warmth = combo_warmth_level(combo)
    max_distance = len(WARMTH_ORDER) - 1
    return 1 - abs(actual_warmth - required_warmth) / max_distance


In [ ]:
def layering_score(combo):
    top, outerwear = combo["top"], combo["outerwear"]
    if outerwear is None:
        return 1.0  # no layering conflict possible

    top_idx = WARMTH_ORDER.index(top["warmth"])
    outer_idx = WARMTH_ORDER.index(outerwear["warmth"])

    if top_idx == 2 and outer_idx == 2:   # heavy under heavy
        return 0.4
    elif top_idx == 2 and outer_idx == 1: # heavy under medium
        return 0.7
    else:
        return 1.0


In [ ]:
FORMALITY_ORDER = ["casual", "athletic", "smart-casual", "formal"]

def occasion_fit_score(combo, occasion_type):
    required = OCCASION_FORMALITY[occasion_type]
    required_idx = FORMALITY_ORDER.index(required)

    items = [combo[k] for k in ["top", "bottom", "outerwear", "shoes"] if combo[k] is not None]
    distances = [abs(FORMALITY_ORDER.index(i["formality"]) - required_idx) for i in items]
    avg_distance = sum(distances) / len(distances)
    return 1 - avg_distance / (len(FORMALITY_ORDER) - 1)


In [ ]:
from datetime import datetime

COOLDOWN_DAYS = 7

def recency_penalty(item):
    if item is None or item.get("last_worn_at") is None:
        return 0.0
    last_worn = item["last_worn_at"]
    if isinstance(last_worn, str):
        last_worn = datetime.fromisoformat(last_worn)
    days_since = (datetime.now() - last_worn).days
    return max(0, 1 - days_since / COOLDOWN_DAYS)


In [ ]:
WEIGHTS = {
    "color": 1.0,
    "temp": 1.5,
    "layering": 0.8,
    "occasion": 1.2,
    "recency": 1.0
}

def score_combo(combo, weather, occasion_type):
    top, bottom = combo["top"], combo["bottom"]

    color_score = color_compatibility_score(top["color_primary"], bottom["color_primary"])
    temp_score = temperature_fit_score(combo, weather)
    layer_score = layering_score(combo)
    occ_score = occasion_fit_score(combo, occasion_type)

    recency = sum(recency_penalty(combo[k]) for k in ["top", "bottom", "outerwear", "shoes"])

    total = (
        WEIGHTS["color"] * color_score +
        WEIGHTS["temp"] * temp_score +
        WEIGHTS["layering"] * layer_score +
        WEIGHTS["occasion"] * occ_score -
        WEIGHTS["recency"] * recency
    )
    return total


In [ ]:
OVERLAP_PENALTY = 0.5
TIE_THRESHOLD = 0.05

def select_top_combos(combos, weather, occasion_type, n=2):
    if not combos:
        return []

    scored = [(combo, score_combo(combo, weather, occasion_type)) for combo in combos]
    scored.sort(key=lambda x: x[1], reverse=True)

    selected = [scored[0]]
    for combo, score in scored[1:]:
        if len(selected) >= n:
            break
        overlaps = any(combo["top"]["id"] == s[0]["top"]["id"] for s in selected)
        adjusted_score = score - OVERLAP_PENALTY if overlaps else score
        selected.append((combo, adjusted_score))

    selected.sort(key=lambda x: x[1], reverse=True)
    return [s[0] for s in selected[:n]]


In [ ]:
import pandas as pd
from datetime import datetime

# Redefine the recency_penalty function to handle NaN values
COOLDOWN_DAYS = 7 # This constant is used in recency_penalty, so it needs to be available or redefined.

def recency_penalty(item):
    if item is None:
        return 0.0
    last_worn = item.get("last_worn_at")

    # Check for None or NaN values. pd.isna handles both None and numpy.nan (float('nan'))
    if pd.isna(last_worn):
        return 0.0

    if isinstance(last_worn, str):
        # Handle cases where it might be an empty string or 'None' string
        if not last_worn or last_worn.lower() == 'none':
            return 0.0
        try:
            last_worn = datetime.fromisoformat(last_worn)
        except ValueError: # If it's a string but not a valid ISO format
            return 0.0
    elif not isinstance(last_worn, datetime):
        # If it's neither None/NaN, str, nor datetime, it's an unexpected type.
        # Treat it as if it means no recent wear.
        return 0.0

    days_since = (datetime.now() - last_worn).days
    return max(0, 1 - days_since / COOLDOWN_DAYS)


final_outfits = {}

for name, data in results.items():
    top_combos = select_top_combos(data["combos"], data["weather"], name)
    final_outfits[name] = top_combos

    print(f"\n{name}:")
    for i, combo in enumerate(top_combos, 1):
        outer = combo["outerwear"]["subcategory"] if combo["outerwear"] else "no outerwear"
        print(f"  Option {i}: {combo['top']['subcategory']} + {combo['bottom']['subcategory']} "
              f"+ {outer} + {combo['shoes']['subcategory']}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def display_combo(combo, title=""):
    items = [combo["top"], combo["bottom"], combo["outerwear"], combo["shoes"]]
    labels = ["Top", "Bottom", "Outerwear", "Shoes"]

    # Filter out None (e.g. no outerwear)
    present = [(item, label) for item, label in zip(items, labels) if item is not None]

    fig, axes = plt.subplots(1, len(present), figsize=(4 * len(present), 4))
    if len(present) == 1:
        axes = [axes]

    for ax, (item, label) in zip(axes, present):
        img = Image.open(item["image_path"])
        ax.imshow(img)
        ax.set_title(f"{label}\n{item['category']} ({item['color_primary']})")
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
for name, top_combos in final_outfits.items():
    print(f"\n=== {name.upper()} ===")
    for i, combo in enumerate(top_combos, 1):
        display_combo(combo, title=f"{name} — Option {i}")